## ステップ3 — Unsloth QLoRA + DeepAgents + Turn-Level GRPO

### 1. インストール

In [ ]:
%%bash
pip install -U \
  "unsloth>=2026.9.7" \
  datasets \
  langchain \
  langchain-core \
  langchain-mcp-adapters \
  deepagents \
  packaging \
  rich

In [ ]:
# from dotenv import load_dotenv
# load_dotenv(".env", override=True)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = "<HF_TOKEN>"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

### 2. データセット

In [ ]:
%%writefile deepwiki_grpo_dataset_300.jsonl
{"id": "deepwiki-0001", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "architecture", "must_mention": ["RL-Factory", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0002", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "execution_flow", "must_mention": ["RL-Factory", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0003", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "dependencies", "must_mention": ["RL-Factory", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0004", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "configuration", "must_mention": ["RL-Factory", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0005", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "testing", "must_mention": ["RL-Factory", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0006", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "training_or_core_algorithm", "must_mention": ["RL-Factory", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0007", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "extension_points", "must_mention": ["RL-Factory", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0008", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "debugging", "must_mention": ["RL-Factory", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0009", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "api_comparison", "must_mention": ["RL-Factory", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0010", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "onboarding", "must_mention": ["RL-Factory", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0011", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "architecture", "must_mention": ["trl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0012", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "execution_flow", "must_mention": ["trl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0013", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "dependencies", "must_mention": ["trl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0014", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "configuration", "must_mention": ["trl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0015", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "testing", "must_mention": ["trl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0016", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "training_or_core_algorithm", "must_mention": ["trl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0017", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "extension_points", "must_mention": ["trl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0018", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "debugging", "must_mention": ["trl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0019", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "api_comparison", "must_mention": ["trl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0020", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "onboarding", "must_mention": ["trl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0021", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "architecture", "must_mention": ["transformers", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0022", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "execution_flow", "must_mention": ["transformers", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0023", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "dependencies", "must_mention": ["transformers", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0024", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "configuration", "must_mention": ["transformers", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0025", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "testing", "must_mention": ["transformers", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0026", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "training_or_core_algorithm", "must_mention": ["transformers", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0027", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "extension_points", "must_mention": ["transformers", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0028", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "debugging", "must_mention": ["transformers", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0029", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "api_comparison", "must_mention": ["transformers", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0030", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "onboarding", "must_mention": ["transformers", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0031", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "architecture", "must_mention": ["peft", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0032", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "execution_flow", "must_mention": ["peft", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0033", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "dependencies", "must_mention": ["peft", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0034", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "configuration", "must_mention": ["peft", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0035", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "testing", "must_mention": ["peft", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0036", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "training_or_core_algorithm", "must_mention": ["peft", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0037", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "extension_points", "must_mention": ["peft", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0038", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "debugging", "must_mention": ["peft", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0039", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "api_comparison", "must_mention": ["peft", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0040", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "onboarding", "must_mention": ["peft", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0041", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "architecture", "must_mention": ["accelerate", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0042", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "execution_flow", "must_mention": ["accelerate", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0043", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "dependencies", "must_mention": ["accelerate", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0044", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "configuration", "must_mention": ["accelerate", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0045", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "testing", "must_mention": ["accelerate", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0046", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "training_or_core_algorithm", "must_mention": ["accelerate", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0047", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "extension_points", "must_mention": ["accelerate", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0048", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "debugging", "must_mention": ["accelerate", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0049", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "api_comparison", "must_mention": ["accelerate", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0050", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "onboarding", "must_mention": ["accelerate", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0051", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "architecture", "must_mention": ["datasets", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0052", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "execution_flow", "must_mention": ["datasets", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0053", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "dependencies", "must_mention": ["datasets", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0054", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "configuration", "must_mention": ["datasets", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0055", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "testing", "must_mention": ["datasets", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0056", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "training_or_core_algorithm", "must_mention": ["datasets", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0057", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "extension_points", "must_mention": ["datasets", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0058", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "debugging", "must_mention": ["datasets", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0059", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "api_comparison", "must_mention": ["datasets", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0060", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "onboarding", "must_mention": ["datasets", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0061", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "architecture", "must_mention": ["unsloth", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0062", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "execution_flow", "must_mention": ["unsloth", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0063", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "dependencies", "must_mention": ["unsloth", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0064", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "configuration", "must_mention": ["unsloth", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0065", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "testing", "must_mention": ["unsloth", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0066", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "training_or_core_algorithm", "must_mention": ["unsloth", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0067", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "extension_points", "must_mention": ["unsloth", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0068", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "debugging", "must_mention": ["unsloth", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0069", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "api_comparison", "must_mention": ["unsloth", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0070", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "onboarding", "must_mention": ["unsloth", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0071", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "architecture", "must_mention": ["langchain", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0072", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "execution_flow", "must_mention": ["langchain", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0073", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "dependencies", "must_mention": ["langchain", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0074", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "configuration", "must_mention": ["langchain", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0075", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "testing", "must_mention": ["langchain", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0076", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "training_or_core_algorithm", "must_mention": ["langchain", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0077", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "extension_points", "must_mention": ["langchain", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0078", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "debugging", "must_mention": ["langchain", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0079", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "api_comparison", "must_mention": ["langchain", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0080", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "onboarding", "must_mention": ["langchain", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0081", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "architecture", "must_mention": ["langgraph", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0082", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "execution_flow", "must_mention": ["langgraph", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0083", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "dependencies", "must_mention": ["langgraph", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0084", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "configuration", "must_mention": ["langgraph", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0085", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "testing", "must_mention": ["langgraph", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0086", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "training_or_core_algorithm", "must_mention": ["langgraph", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0087", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "extension_points", "must_mention": ["langgraph", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0088", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "debugging", "must_mention": ["langgraph", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0089", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "api_comparison", "must_mention": ["langgraph", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0090", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "onboarding", "must_mention": ["langgraph", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0091", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "architecture", "must_mention": ["deepagents", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0092", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "execution_flow", "must_mention": ["deepagents", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0093", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "dependencies", "must_mention": ["deepagents", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0094", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "configuration", "must_mention": ["deepagents", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0095", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "testing", "must_mention": ["deepagents", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0096", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "training_or_core_algorithm", "must_mention": ["deepagents", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0097", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "extension_points", "must_mention": ["deepagents", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0098", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "debugging", "must_mention": ["deepagents", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0099", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "api_comparison", "must_mention": ["deepagents", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0100", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "onboarding", "must_mention": ["deepagents", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0101", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "architecture", "must_mention": ["vllm", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0102", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "execution_flow", "must_mention": ["vllm", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0103", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "dependencies", "must_mention": ["vllm", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0104", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "configuration", "must_mention": ["vllm", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0105", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "testing", "must_mention": ["vllm", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0106", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "training_or_core_algorithm", "must_mention": ["vllm", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0107", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "extension_points", "must_mention": ["vllm", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0108", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "debugging", "must_mention": ["vllm", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0109", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "api_comparison", "must_mention": ["vllm", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0110", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "onboarding", "must_mention": ["vllm", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0111", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "architecture", "must_mention": ["sglang", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0112", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "execution_flow", "must_mention": ["sglang", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0113", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "dependencies", "must_mention": ["sglang", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0114", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "configuration", "must_mention": ["sglang", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0115", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "testing", "must_mention": ["sglang", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0116", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "training_or_core_algorithm", "must_mention": ["sglang", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0117", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "extension_points", "must_mention": ["sglang", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0118", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "debugging", "must_mention": ["sglang", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0119", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "api_comparison", "must_mention": ["sglang", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0120", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "onboarding", "must_mention": ["sglang", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0121", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "architecture", "must_mention": ["Megatron-LM", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0122", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "execution_flow", "must_mention": ["Megatron-LM", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0123", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "dependencies", "must_mention": ["Megatron-LM", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0124", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "configuration", "must_mention": ["Megatron-LM", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0125", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "testing", "must_mention": ["Megatron-LM", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0126", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "training_or_core_algorithm", "must_mention": ["Megatron-LM", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0127", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "extension_points", "must_mention": ["Megatron-LM", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0128", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "debugging", "must_mention": ["Megatron-LM", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0129", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "api_comparison", "must_mention": ["Megatron-LM", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0130", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "onboarding", "must_mention": ["Megatron-LM", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0131", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "architecture", "must_mention": ["DeepSpeed", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0132", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "execution_flow", "must_mention": ["DeepSpeed", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0133", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "dependencies", "must_mention": ["DeepSpeed", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0134", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "configuration", "must_mention": ["DeepSpeed", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0135", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "testing", "must_mention": ["DeepSpeed", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0136", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "training_or_core_algorithm", "must_mention": ["DeepSpeed", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0137", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "extension_points", "must_mention": ["DeepSpeed", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0138", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "debugging", "must_mention": ["DeepSpeed", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0139", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "api_comparison", "must_mention": ["DeepSpeed", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0140", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "onboarding", "must_mention": ["DeepSpeed", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0141", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "architecture", "must_mention": ["ray", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0142", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "execution_flow", "must_mention": ["ray", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0143", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "dependencies", "must_mention": ["ray", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0144", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "configuration", "must_mention": ["ray", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0145", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "testing", "must_mention": ["ray", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0146", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "training_or_core_algorithm", "must_mention": ["ray", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0147", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "extension_points", "must_mention": ["ray", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0148", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "debugging", "must_mention": ["ray", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0149", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "api_comparison", "must_mention": ["ray", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0150", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "onboarding", "must_mention": ["ray", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0151", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "architecture", "must_mention": ["pytorch", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0152", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "execution_flow", "must_mention": ["pytorch", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0153", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "dependencies", "must_mention": ["pytorch", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0154", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "configuration", "must_mention": ["pytorch", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0155", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "testing", "must_mention": ["pytorch", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0156", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "training_or_core_algorithm", "must_mention": ["pytorch", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0157", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "extension_points", "must_mention": ["pytorch", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0158", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "debugging", "must_mention": ["pytorch", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0159", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "api_comparison", "must_mention": ["pytorch", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0160", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "onboarding", "must_mention": ["pytorch", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0161", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "architecture", "must_mention": ["jax", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0162", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "execution_flow", "must_mention": ["jax", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0163", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "dependencies", "must_mention": ["jax", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0164", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "configuration", "must_mention": ["jax", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0165", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "testing", "must_mention": ["jax", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0166", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "training_or_core_algorithm", "must_mention": ["jax", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0167", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "extension_points", "must_mention": ["jax", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0168", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "debugging", "must_mention": ["jax", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0169", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "api_comparison", "must_mention": ["jax", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0170", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "onboarding", "must_mention": ["jax", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0171", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "architecture", "must_mention": ["openai-python", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0172", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "execution_flow", "must_mention": ["openai-python", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0173", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "dependencies", "must_mention": ["openai-python", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0174", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "configuration", "must_mention": ["openai-python", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0175", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "testing", "must_mention": ["openai-python", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0176", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "training_or_core_algorithm", "must_mention": ["openai-python", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0177", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "extension_points", "must_mention": ["openai-python", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0178", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "debugging", "must_mention": ["openai-python", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0179", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "api_comparison", "must_mention": ["openai-python", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0180", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "onboarding", "must_mention": ["openai-python", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0181", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "architecture", "must_mention": ["ollama", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0182", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "execution_flow", "must_mention": ["ollama", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0183", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "dependencies", "must_mention": ["ollama", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0184", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "configuration", "must_mention": ["ollama", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0185", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "testing", "must_mention": ["ollama", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0186", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "training_or_core_algorithm", "must_mention": ["ollama", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0187", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "extension_points", "must_mention": ["ollama", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0188", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "debugging", "must_mention": ["ollama", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0189", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "api_comparison", "must_mention": ["ollama", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0190", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "onboarding", "must_mention": ["ollama", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0191", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "architecture", "must_mention": ["llama.cpp", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0192", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "execution_flow", "must_mention": ["llama.cpp", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0193", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "dependencies", "must_mention": ["llama.cpp", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0194", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "configuration", "must_mention": ["llama.cpp", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0195", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "testing", "must_mention": ["llama.cpp", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0196", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "training_or_core_algorithm", "must_mention": ["llama.cpp", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0197", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "extension_points", "must_mention": ["llama.cpp", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0198", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "debugging", "must_mention": ["llama.cpp", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0199", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "api_comparison", "must_mention": ["llama.cpp", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0200", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "onboarding", "must_mention": ["llama.cpp", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0201", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "architecture", "must_mention": ["litgpt", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0202", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "execution_flow", "must_mention": ["litgpt", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0203", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "dependencies", "must_mention": ["litgpt", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0204", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "configuration", "must_mention": ["litgpt", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0205", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "testing", "must_mention": ["litgpt", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0206", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "training_or_core_algorithm", "must_mention": ["litgpt", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0207", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "extension_points", "must_mention": ["litgpt", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0208", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "debugging", "must_mention": ["litgpt", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0209", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "api_comparison", "must_mention": ["litgpt", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0210", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "onboarding", "must_mention": ["litgpt", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0211", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "architecture", "must_mention": ["axolotl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0212", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "execution_flow", "must_mention": ["axolotl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0213", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "dependencies", "must_mention": ["axolotl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0214", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "configuration", "must_mention": ["axolotl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0215", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "testing", "must_mention": ["axolotl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0216", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "training_or_core_algorithm", "must_mention": ["axolotl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0217", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "extension_points", "must_mention": ["axolotl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0218", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "debugging", "must_mention": ["axolotl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0219", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "api_comparison", "must_mention": ["axolotl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0220", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "onboarding", "must_mention": ["axolotl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0221", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "architecture", "must_mention": ["OpenRLHF", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0222", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "execution_flow", "must_mention": ["OpenRLHF", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0223", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "dependencies", "must_mention": ["OpenRLHF", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0224", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "configuration", "must_mention": ["OpenRLHF", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0225", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "testing", "must_mention": ["OpenRLHF", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0226", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "training_or_core_algorithm", "must_mention": ["OpenRLHF", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0227", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "extension_points", "must_mention": ["OpenRLHF", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0228", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "debugging", "must_mention": ["OpenRLHF", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0229", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "api_comparison", "must_mention": ["OpenRLHF", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0230", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "onboarding", "must_mention": ["OpenRLHF", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0231", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "architecture", "must_mention": ["verl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0232", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "execution_flow", "must_mention": ["verl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0233", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "dependencies", "must_mention": ["verl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0234", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "configuration", "must_mention": ["verl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0235", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "testing", "must_mention": ["verl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0236", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "training_or_core_algorithm", "must_mention": ["verl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0237", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "extension_points", "must_mention": ["verl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0238", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "debugging", "must_mention": ["verl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0239", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "api_comparison", "must_mention": ["verl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0240", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "onboarding", "must_mention": ["verl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0241", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "architecture", "must_mention": ["open-instruct", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0242", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "execution_flow", "must_mention": ["open-instruct", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0243", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "dependencies", "must_mention": ["open-instruct", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0244", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "configuration", "must_mention": ["open-instruct", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0245", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "testing", "must_mention": ["open-instruct", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0246", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "training_or_core_algorithm", "must_mention": ["open-instruct", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0247", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "extension_points", "must_mention": ["open-instruct", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0248", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "debugging", "must_mention": ["open-instruct", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0249", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "api_comparison", "must_mention": ["open-instruct", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0250", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "onboarding", "must_mention": ["open-instruct", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0251", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "architecture", "must_mention": ["mlflow", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0252", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "execution_flow", "must_mention": ["mlflow", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0253", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "dependencies", "must_mention": ["mlflow", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0254", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "configuration", "must_mention": ["mlflow", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0255", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "testing", "must_mention": ["mlflow", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0256", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "training_or_core_algorithm", "must_mention": ["mlflow", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0257", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "extension_points", "must_mention": ["mlflow", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0258", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "debugging", "must_mention": ["mlflow", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0259", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "api_comparison", "must_mention": ["mlflow", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0260", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "onboarding", "must_mention": ["mlflow", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0261", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "architecture", "must_mention": ["wandb", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0262", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "execution_flow", "must_mention": ["wandb", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0263", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "dependencies", "must_mention": ["wandb", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0264", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "configuration", "must_mention": ["wandb", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0265", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "testing", "must_mention": ["wandb", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0266", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "training_or_core_algorithm", "must_mention": ["wandb", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0267", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "extension_points", "must_mention": ["wandb", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0268", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "debugging", "must_mention": ["wandb", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0269", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "api_comparison", "must_mention": ["wandb", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0270", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "onboarding", "must_mention": ["wandb", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0271", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "architecture", "must_mention": ["airflow", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0272", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "execution_flow", "must_mention": ["airflow", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0273", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "dependencies", "must_mention": ["airflow", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0274", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "configuration", "must_mention": ["airflow", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0275", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "testing", "must_mention": ["airflow", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0276", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "training_or_core_algorithm", "must_mention": ["airflow", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0277", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "extension_points", "must_mention": ["airflow", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0278", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "debugging", "must_mention": ["airflow", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0279", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "api_comparison", "must_mention": ["airflow", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0280", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "onboarding", "must_mention": ["airflow", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0281", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "architecture", "must_mention": ["fastapi", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0282", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "execution_flow", "must_mention": ["fastapi", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0283", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "dependencies", "must_mention": ["fastapi", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0284", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "configuration", "must_mention": ["fastapi", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0285", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "testing", "must_mention": ["fastapi", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0286", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "training_or_core_algorithm", "must_mention": ["fastapi", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0287", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "extension_points", "must_mention": ["fastapi", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0288", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "debugging", "must_mention": ["fastapi", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0289", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "api_comparison", "must_mention": ["fastapi", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0290", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "onboarding", "must_mention": ["fastapi", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0291", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "architecture", "must_mention": ["pytest", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0292", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "execution_flow", "must_mention": ["pytest", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0293", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "dependencies", "must_mention": ["pytest", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0294", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "configuration", "must_mention": ["pytest", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0295", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "testing", "must_mention": ["pytest", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0296", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "training_or_core_algorithm", "must_mention": ["pytest", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0297", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "extension_points", "must_mention": ["pytest", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0298", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "debugging", "must_mention": ["pytest", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0299", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "api_comparison", "must_mention": ["pytest", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0300", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "onboarding", "must_mention": ["pytest", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}


### 3. 学習コード

In [ ]:
# ============================================================
# STEP 3
# Unsloth QLoRA + DeepAgents + Turn-Level GRPO
# 初期ベースモデルから学習する Level-3 multi-turn GRPO
# Step2 の OOM / hard truncate 対策を維持し、turn-level credit assignment を追加
# ============================================================

import asyncio
import concurrent.futures
import gc
import importlib.metadata
import inspect
import json
import os
import random
import re
import sys
import uuid
from dataclasses import dataclass, field
from typing import Any, Optional, Sequence

import unsloth
import bitsandbytes
import torch
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from deepagents import create_deep_agent
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import BaseTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from packaging.version import Version
from pydantic import PrivateAttr
from unsloth import FastLanguageModel


# ============================================================
# 0. 実行時の設定
# ============================================================

# CUDA アロケータの断片化対策が主な解決策ではないが、可変長のエージェント処理には拡張可能なセグメントが役立つ。
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ベースモデル。このバージョンでは初期モデルから学習し、ステップ1のアダプタは読み込まない。
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# この学習用実行環境で使うモデルのコンテキスト長の上限。
# DeepAgents は複数ターンにわたる長いコンテキストを作るため、4096 は A100 80GB 向けの控えめな値となる。
MAX_SEQ_LENGTH = 6144

# アシスタントの1ターンで生成するトークンの最大数。
# 入力長と MAX_NEW_TOKENS の合計が MAX_SEQ_LENGTH を超えないよう、生成時の入力を強制的に切り詰める。
MAX_NEW_TOKENS = 1024

# システムプロンプトとツールのスキーマが含まれるため、プロンプトの先頭を保持する。
TRUNCATION_HEAD_TOKENS = 1024

# 勾配計算を伴う GRPO 更新で使う出力トークンのチャンクサイズ。
# 小さくすると lm_head／logits のピークメモリを減らせるが、計算時間は増える。
TRAIN_LOGPROB_CHUNK_SIZE = 64

# GRPO のグループサイズ。
NUM_GENERATIONS = 4

# 学習エポック数。
NUM_EPOCHS = 2

# ロールアウト時のサンプリング設定。
ROLLOUT_TEMPERATURE = 0.8
ROLLOUT_TOP_P = 0.95

# GRPO／PPO のクリップ幅。
CLIP_EPSILON = 0.2

# 参照モデルとの KL 正則化係数。
# 参照ポリシーの順伝播により更新時の計算量がほぼ倍増するため、最初の安定性確認では 0.0 にする。
# このステップ2はベースモデルから開始するので、後から BETA を有効にする場合は disable_adapter() を参照モデルとして使える。
BETA = 0.0

# Turn-level return-to-go の割引率。短い repository research ではまず 1.0 を推奨する。
TURN_RETURN_GAMMA = 1.0

# Leave-one-out advantage の極端な外れ値を抑える。None なら clipping を無効化する。
TURN_ADVANTAGE_CLIP = 5.0

# 1 assistant turn 内の tool call は4件まで追加 penalty なしとし、それ以上は軽く減点する。
MAX_FREE_TOOL_CALLS_PER_TURN = 4
EXTRA_TOOL_CALL_PENALTY = -0.05


# オプティマイザの設定。
LEARNING_RATE = 1e-5
MAX_GRAD_NORM = 1.0

# チェックポイントの設定。
SAVE_EVERY = 25
OUTPUT_DIR = "./outputs/step3-unsloth-deepagents-turn-grpo"
FINAL_PATH = OUTPUT_DIR + "/final"

# 再現性のための乱数シード。
SEED = 42

# 任意の簡易動作確認用のサンプル数上限。学習用データ全体を使う場合は None にする。
DEBUG_MAX_TRAIN_SAMPLES = None

# 学習ステップごとに GPU メモリの詳細を表示する。
PRINT_GPU_MEMORY = True


# ============================================================
# 1. バージョンと必要な機能の確認
# ============================================================

# 実行時の依存ライブラリのバージョンを収集して、互換性確認の対象を明示する。
PYTHON_VERSION = Version(str(sys.version_info.major) + "." + str(sys.version_info.minor) + "." + str(sys.version_info.micro))
TORCH_VERSION = Version(torch.__version__.split("+")[0])
TRANSFORMERS_VERSION = Version(transformers.__version__)
UNSLOTH_VERSION = Version(unsloth.__version__)
BITSANDBYTES_VERSION = Version(bitsandbytes.__version__)
DEEPAGENTS_VERSION = importlib.metadata.version("deepagents")
LANGCHAIN_CORE_VERSION = importlib.metadata.version("langchain-core")
MCP_ADAPTERS_VERSION = importlib.metadata.version("langchain-mcp-adapters")

# 確認したバージョンを表示し、環境差による動作不一致を追跡できるようにする。
print("python                 :", PYTHON_VERSION)
print("torch                  :", TORCH_VERSION)
print("transformers           :", TRANSFORMERS_VERSION)
print("unsloth                :", UNSLOTH_VERSION)
print("bitsandbytes           :", BITSANDBYTES_VERSION)
print("deepagents             :", DEEPAGENTS_VERSION)
print("langchain-core         :", LANGCHAIN_CORE_VERSION)
print("langchain-mcp-adapters :", MCP_ADAPTERS_VERSION)

# Qwen3 の省メモリ logits 計算と 4-bit QLoRA に必要なバージョン・GPU 能力を早期に検査する。
assert PYTHON_VERSION >= Version("3.10.0"), "Python 3.10 or later is required."
assert TORCH_VERSION >= Version("2.4.0"), "PyTorch 2.4 or later is required."
assert TRANSFORMERS_VERSION >= Version("4.51.0"), "Transformers 4.51 or later is required for Qwen3 logits_to_keep support."
assert UNSLOTH_VERSION >= Version("2026.9.7"), "This code targets Unsloth 2026.9.7 or later."
assert BITSANDBYTES_VERSION >= Version("0.43.0"), "bitsandbytes 0.43 or later is required."
assert torch.cuda.is_available(), "CUDA GPU is required."
assert torch.cuda.is_bf16_supported(), "A BF16-capable GPU is required."
assert hasattr(FastLanguageModel, "from_pretrained"), "FastLanguageModel.from_pretrained is unavailable."
assert hasattr(FastLanguageModel, "get_peft_model"), "FastLanguageModel.get_peft_model is unavailable."
assert callable(create_deep_agent), "create_deep_agent is unavailable."

# 生成枠・チャンク長・同一プロンプトのロールアウト数が学習の前提を満たすことを確認する。
assert MAX_NEW_TOKENS < MAX_SEQ_LENGTH, "MAX_NEW_TOKENS must be smaller than MAX_SEQ_LENGTH."
assert TRAIN_LOGPROB_CHUNK_SIZE > 0, "TRAIN_LOGPROB_CHUNK_SIZE must be positive."
assert NUM_GENERATIONS >= 2, "Turn-level group-relative learning requires at least two rollouts per prompt."
assert 0.0 < TURN_RETURN_GAMMA <= 1.0, "TURN_RETURN_GAMMA must be in (0, 1]."

print("Version and capability assertions: OK")


# ============================================================
# 2. 補助関数
# ============================================================

def cleanup_cuda():
    """大きな処理の合間に Python の参照とキャッシュ済みの CUDA メモリブロックを解放する。"""
    # 参照されなくなった Python オブジェクトを回収した後、未使用の CUDA キャッシュを返す。
    gc.collect()
    torch.cuda.empty_cache()


def gpu_memory_snapshot(label):
    """GPU メモリの使用量・予約量・ピーク値を GiB 単位で表示する。"""
    # 診断を無効にした場合は CUDA メモリの取得を省略する。
    if not PRINT_GPU_MEMORY:
        return

    # 使用中・予約中・ピークを区別して、メモリ不足の発生箇所を調べやすくする。
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"[GPU] {label}: allocated={allocated:.2f} GiB reserved={reserved:.2f} GiB peak={peak:.2f} GiB")


def run_async_sync(coro):
    """スクリプトと Jupyter Notebook のどちらからも非同期コルーチンを実行する。"""
    # Notebook には実行中のイベントループがあるため、同期実行の方法を切り替える。
    try:
        asyncio.get_running_loop()
        has_loop = True
    except RuntimeError:
        has_loop = False

    if not has_loop:
        return asyncio.run(coro)

    # 既存のループ内では別スレッドで専用ループを立ち上げ、結果を同期的に受け取る。
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        future = pool.submit(asyncio.run, coro)
        return future.result()


def contains_any(text, candidates):
    """ジェネレータ式を使わず、候補の文字列がいずれか1つでも含まれるか調べる。"""
    # 見つかった時点で探索を終了し、見つからなければ False を返す。
    for candidate in candidates:
        if candidate in text:
            return True
    return False


def count_true_strings_in_text(text, strings):
    """テキストに出現する対象文字列の数を数える。"""
    count = 0
    # 各対象文字列の包含を判定する。出現回数ではなく一致した項目数を数える。
    for item in strings:
        if str(item).lower() in text:
            count += 1
    return count


def count_trainable_parameters(module):
    """内包表記やジェネレータ式を使わず、学習可能なパラメータ数と総数を数える。"""
    trainable = 0
    total = 0
    # 総パラメータ数と、勾配更新されるパラメータ数を別々に集計する。
    for parameter in module.parameters():
        number = parameter.numel()
        total += number
        if parameter.requires_grad:
            trainable += number
    return trainable, total


def model_is_loaded_in_4bit(module):
    """PEFT／Unsloth の一般的なラッパー階層をたどり、4ビット読み込みのフラグを確認する。"""
    current = module
    # フラグがモデル本体ではなくラッパー内にある場合も、最大5階層まで探索する。
    for _ in range(5):
        if getattr(current, "is_loaded_in_4bit", False):
            return True
        next_model = getattr(current, "base_model", None)
        if next_model is None:
            next_model = getattr(current, "model", None)
        if next_model is None or next_model is current:
            break
        current = next_model
    return False


# ============================================================
# 3. MCP の定義
# ============================================================

# DeepWiki の MCP サーバーに接続し、実際のエージェントが使用するツール群を取得する。
mcp_client = MultiServerMCPClient({"deepwiki": {"url": "https://mcp.deepwiki.com/mcp", "transport": "streamable_http"}})
deepwiki_tools = run_async_sync(mcp_client.get_tools())

# 使用可能なツール名を表示し、ロールアウト前に取得結果を確認する。
print("MCP tools:")
for tool in deepwiki_tools:
    print(" -", tool.name)

assert len(deepwiki_tools) > 0, "No DeepWiki MCP tools were loaded."


# ============================================================
# 4. データセットの読み込み
# ============================================================

# JSONL 全体を読み込み、split 列によって学習・検証・テストのデータを分ける。
dataset = load_dataset("json", data_files="deepwiki_grpo_dataset_300.jsonl", split="train")
train_dataset = dataset.filter(lambda row: row["split"] == "train")
validation_dataset = dataset.filter(lambda row: row["split"] == "validation")
test_dataset = dataset.filter(lambda row: row["split"] == "test")

# 動作確認時だけ学習データ数を制限し、None の場合は全件を使う。
if DEBUG_MAX_TRAIN_SAMPLES is not None:
    selected_count = min(DEBUG_MAX_TRAIN_SAMPLES, len(train_dataset))
    train_dataset = train_dataset.select(range(selected_count))

print("train      :", len(train_dataset))
print("validation :", len(validation_dataset))
print("test       :", len(test_dataset))

# 報酬とロールアウトに必要な項目がそろい、学習対象が空でないことを確認する。
required_columns = {"id", "prompt", "repository", "must_mention", "evidence_requirements", "reward_spec", "max_tool_steps", "split"}
assert required_columns.issubset(set(train_dataset.column_names)), "Dataset columns are missing required fields."
assert len(train_dataset) > 0, "Training split is empty."


# ============================================================
# 5. 初期ベースモデルからの Unsloth 4ビット QLoRA モデル
# ============================================================

# ここではステップ1のアダプタを意図的に読み込まない。
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None)

# 4ビットのベースモデルに新しい学習可能な LoRA アダプタを追加する。
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none", target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], use_gradient_checkpointing="unsloth", random_state=SEED)

# pad token が未設定なら EOS を割り当て、バッチ化時のパディング ID を確保する。
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# 学習時の順伝播では KV キャッシュを保持しない。
model.config.use_cache = False

# 実際に QLoRA が有効になっているか確認する。
is_4bit = model_is_loaded_in_4bit(model)
trainable_count, total_count = count_trainable_parameters(model)
trainable_ratio = 100.0 * trainable_count / total_count

print("is_loaded_in_4bit:", is_4bit)
print(f"trainable={trainable_count:,} total={total_count:,} ratio={trainable_ratio:.4f}%")

# 4-bit 化と、学習可能なパラメータが少数の LoRA 重みだけであることを検証する。
assert is_4bit, "The base model is not loaded in 4-bit mode. QLoRA is inactive."
assert trainable_count > 0, "No trainable LoRA parameters were found."
assert trainable_count < total_count * 0.1, "Too many parameters are trainable. Check the PEFT configuration."
assert hasattr(model, "disable_adapter"), "PEFT disable_adapter() is required when BETA > 0."

# メモリを節約した対数確率の計算に必要なので、実行時に logits_to_keep の対応状況を確認する。
probe_device = next(model.parameters()).device
probe_token_id = tokenizer.eos_token_id
if probe_token_id is None:
    probe_token_id = tokenizer.pad_token_id
probe_ids = torch.tensor([[probe_token_id, probe_token_id]], dtype=torch.long, device=probe_device)

# 小さな入力で logits_to_keep の挙動を調べ、不要な logits が生成されないことを確かめる。
with torch.no_grad():
    probe_outputs = model(input_ids=probe_ids, use_cache=False, logits_to_keep=2)
assert probe_outputs.logits.shape[1] <= 2, "The model does not honor logits_to_keep. This memory-safe implementation requires it."

# プローブの参照を解放し、学習前の GPU キャッシュを整理する。
del probe_outputs
del probe_ids
cleanup_cuda()

print("Qwen3 logits_to_keep capability: OK")


# ============================================================
# 6. 軌跡の記録
# ============================================================

# 1回のモデル呼び出しの入力・生成結果・旧方策確率を保存し、ターンごとの再計算に使う。
@dataclass
class LLMCallRecord:
    input_ids: list[int]
    output_ids: list[int]
    old_logprobs: list[float]
    input_text: str
    output_text: str
    clean_text: str
    parsed_tool_calls: list[dict]
    had_bound_tools: bool
    original_input_length: int
    used_input_length: int
    truncated_tokens: int


# 1 assistant decision turn と、その直後に実行された tool 結果をまとめる。
@dataclass
class TurnDetail:
    assistant_content: str
    tool_calls: list[dict]
    tool_messages: list[ToolMessage]
    is_final: bool
    local_reward: float = 0.0
    reward_components: dict[str, float] = field(default_factory=dict)


# 1 rollout の turn-level reward / return / advantage まで保持する。
@dataclass
class AgentTrajectory:
    records: list[LLMCallRecord]
    turns: list[TurnDetail]
    final_answer: str
    observations: str
    tool_calls: int
    tool_errors: int
    duplicate_calls: int
    turn_rewards: list[float] = field(default_factory=list)
    returns_to_go: list[float] = field(default_factory=list)
    turn_advantages: list[float] = field(default_factory=list)
    reward_components: dict[str, float] = field(default_factory=dict)
    internal_model_calls: int = 0
    reward: float = 0.0


# ============================================================
# 7. コンテキストの切り詰めと対数確率の補助関数
# ============================================================

def truncate_generation_inputs(inputs, max_seq_length, max_new_tokens, head_tokens):
    """システム文・ツール定義を含む先頭と直近の会話末尾を残し、トークン化済み入力に長さの上限を設ける。"""
    input_ids = inputs["input_ids"]
    original_length = int(input_ids.shape[-1])

    # 生成分を先に予約し、モデルに渡せる入力トークン数の上限を決める。
    max_input_length = max_seq_length - max_new_tokens
    if max_input_length <= 0:
        raise ValueError("The generation budget leaves no room for input tokens.")
    if original_length <= max_input_length:
        return inputs, original_length, original_length, 0

    # システム文・ツールスキーマを含む先頭と、最新のやり取りを含む末尾を優先して残す。
    effective_head_tokens = min(head_tokens, max_input_length // 2)
    tail_tokens = max_input_length - effective_head_tokens
    truncated = {}
    for key in inputs:
        value = inputs[key]
        # input_ids と同じ長さの attention_mask 等にも、同じ切り詰めを適用する。
        if torch.is_tensor(value) and value.ndim >= 2 and int(value.shape[-1]) == original_length:
            head_value = value[..., :effective_head_tokens]
            tail_value = value[..., -tail_tokens:]
            truncated[key] = torch.cat([head_value, tail_value], dim=-1)
        else:
            truncated[key] = value
    used_length = int(truncated["input_ids"].shape[-1])
    removed = original_length - used_length
    return truncated, original_length, used_length, removed


def rollout_logprobs_for_generated(model_object, input_ids, generated_ids):
    """必要な出力位置の logits のみを実体化し、勾配なしでロールアウト時の方策のトークン対数確率を計算する。"""
    generated_length = int(generated_ids.shape[-1])
    if generated_length == 0:
        return []

    # 各生成トークンの確率は、それより前の入力・生成済みトークンを条件として求める。
    full_ids = torch.cat([input_ids, generated_ids], dim=-1)
    logits_to_keep = generated_length + 1
    with torch.no_grad():
        outputs = model_object(input_ids=full_ids, use_cache=False, logits_to_keep=logits_to_keep)
    logits = outputs.logits
    expected_minimum = generated_length + 1
    if int(logits.shape[1]) < expected_minimum:
        raise RuntimeError(f"Old-policy logprob alignment failed: logits={int(logits.shape[1])}, required={expected_minimum}, generated={generated_length}.")

    # 末尾1位置を除く logits と生成トークンを1位置ずらして対応づける。
    token_logits = logits[:, -expected_minimum:-1, :]
    targets = generated_ids
    log_probs = F.log_softmax(token_logits, dim=-1)
    selected = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    # ロールアウト時の旧方策確率だけを CPU に保存し、計算グラフは残さない。
    result = selected[0].detach().cpu().tolist()
    del selected
    del log_probs
    del token_logits
    del logits
    del outputs
    del full_ids
    return result


def chunk_logprobs_with_grad(record, start_index, end_index):
    """lm_head の logits をチャンク長 + 1 位置に抑え、出力トークンの1チャンク分の対数確率を勾配付きで再計算する。"""
    device = next(model.parameters()).device

    # 当該ターンが実際に見た入力と、対象チャンク末尾までの生成プレフィックスを再構成する。
    input_ids = torch.tensor(record.input_ids, dtype=torch.long, device=device).unsqueeze(0)
    output_prefix = torch.tensor(record.output_ids[:end_index], dtype=torch.long, device=device).unsqueeze(0)
    chunk_targets = torch.tensor(record.output_ids[start_index:end_index], dtype=torch.long, device=device).unsqueeze(0)
    full_ids = torch.cat([input_ids, output_prefix], dim=1)
    chunk_length = end_index - start_index

    # 過去のトークンは条件づけに使い、語彙方向の logits は対象チャンクの末尾だけ取得する。
    logits_to_keep = chunk_length + 1
    assert int(full_ids.shape[1]) <= MAX_SEQ_LENGTH, "Training sequence exceeds MAX_SEQ_LENGTH despite rollout truncation."
    outputs = model(input_ids=full_ids, use_cache=False, logits_to_keep=logits_to_keep)
    logits = outputs.logits
    if int(logits.shape[1]) < logits_to_keep:
        raise RuntimeError(f"Current-policy logprob alignment failed: logits={int(logits.shape[1])}, required={logits_to_keep}.")

    # 次トークン予測の位置合わせをして、学習対象トークンの対数確率を抽出する。
    token_logits = logits[:, -logits_to_keep:-1, :]
    log_probs = F.log_softmax(token_logits, dim=-1)
    selected = log_probs.gather(-1, chunk_targets.unsqueeze(-1)).squeeze(-1)

    # 勾配が必要な selected は残し、それ以外の大きな参照を手放す。
    del log_probs
    del token_logits
    del logits
    del outputs
    del full_ids
    del input_ids
    del output_prefix
    del chunk_targets
    return selected[0]


def chunk_reference_logprobs(record, start_index, end_index):
    """KL 正則化が有効な場合、凍結したベースモデルで1チャンク分の参照対数確率を計算する。"""
    device = next(model.parameters()).device

    # 現在方策と同じターンの入力・生成プレフィックスを参照方策にも与える。
    input_ids = torch.tensor(record.input_ids, dtype=torch.long, device=device).unsqueeze(0)
    output_prefix = torch.tensor(record.output_ids[:end_index], dtype=torch.long, device=device).unsqueeze(0)
    chunk_targets = torch.tensor(record.output_ids[start_index:end_index], dtype=torch.long, device=device).unsqueeze(0)
    full_ids = torch.cat([input_ids, output_prefix], dim=1)
    chunk_length = end_index - start_index
    logits_to_keep = chunk_length + 1
    assert int(full_ids.shape[1]) <= MAX_SEQ_LENGTH, "Reference sequence exceeds MAX_SEQ_LENGTH despite rollout truncation."

    # LoRA を無効化したベースモデルを参照方策とし、勾配は計算しない。
    with model.disable_adapter():
        with torch.no_grad():
            outputs = model(input_ids=full_ids, use_cache=False, logits_to_keep=logits_to_keep)
    logits = outputs.logits
    if int(logits.shape[1]) < logits_to_keep:
        raise RuntimeError(f"Reference-policy logprob alignment failed: logits={int(logits.shape[1])}, required={logits_to_keep}.")

    # 現在方策と同じ位置・トークンを抽出して KL 項の比較対象を揃える。
    token_logits = logits[:, -logits_to_keep:-1, :]
    log_probs = F.log_softmax(token_logits, dim=-1)
    selected = log_probs.gather(-1, chunk_targets.unsqueeze(-1)).squeeze(-1)
    result = selected[0].detach()
    del log_probs
    del token_logits
    del logits
    del outputs
    del full_ids
    del input_ids
    del output_prefix
    del chunk_targets
    return result


# ============================================================
# 8. DeepAgents 互換の記録機能付きチャットモデル
# ============================================================

class DeepAgentsPolicyModel(BaseChatModel):
    model: Any
    tokenizer: Any
    max_seq_length: int = MAX_SEQ_LENGTH
    max_new_tokens: int = MAX_NEW_TOKENS
    truncation_head_tokens: int = TRUNCATION_HEAD_TOKENS
    temperature: float = ROLLOUT_TEMPERATURE
    top_p: float = ROLLOUT_TOP_P
    _bound_tools: list[dict] = PrivateAttr(default_factory=list)
    _records: list[LLMCallRecord] = PrivateAttr(default_factory=list)

    @property
    def _llm_type(self):
        # LangChain に返すチャットモデルの識別名。
        return "unsloth-deepagents-grpo-oom-safe"

    def clear_records(self):
        # 別の軌跡に前のモデル呼び出し履歴が混ざらないようにする。
        self._records.clear()

    @property
    def records(self):
        # 更新対象のモデル呼び出し履歴をロールアウト側から参照させる。
        return self._records

    def bind_tools(self, tools: Sequence[dict | type | BaseTool | Any], *, tool_choice: Optional[Any] = None, **kwargs):
        schemas = []

        # DeepAgents が渡したツール定義を HF chat template 用のスキーマに変換する。
        for tool in tools:
            schemas.append(convert_to_openai_tool(tool))
        bound = self.model_copy(deep=False)
        bound._bound_tools = schemas

        # bind 後の複製と元のモデルで同じ記録リストを共有する。
        bound._records = self._records
        return bound

    @staticmethod
    def _content_to_text(content):
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            texts = []

            # 複数ブロック形式のメッセージから文字列・text ブロックのみを取り出す。
            for block in content:
                if isinstance(block, str):
                    texts.append(block)
                elif isinstance(block, dict) and block.get("type") == "text":
                    texts.append(block.get("text", ""))
            return "\n".join(texts)
        return str(content)

    def _convert_messages(self, messages):
        output = []

        # LangChain の型ごとに HF の role・content・tool_calls 形式へ変換する。
        for msg in messages:
            if isinstance(msg, SystemMessage):
                output.append({"role": "system", "content": self._content_to_text(msg.content)})
            elif isinstance(msg, HumanMessage):
                output.append({"role": "user", "content": self._content_to_text(msg.content)})
            elif isinstance(msg, AIMessage):
                item = {"role": "assistant", "content": self._content_to_text(msg.content)}
                if msg.tool_calls:
                    # 呼び出し名と引数を、次ターンの chat template に再投入できる形で保持する。
                    item["tool_calls"] = []
                    for call in msg.tool_calls:
                        item["tool_calls"].append({"type": "function", "function": {"name": call["name"], "arguments": call["args"]}})
                output.append(item)
            elif isinstance(msg, ToolMessage):
                item = {"role": "tool", "content": self._content_to_text(msg.content)}
                if getattr(msg, "tool_call_id", None):
                    item["tool_call_id"] = msg.tool_call_id
                if getattr(msg, "name", None):
                    item["name"] = msg.name
                output.append(item)
        return output

    @staticmethod
    def _parse_tool_calls(text):
        result = []

        # Qwen 形式の各 <tool_call> ブロックを抽出する。解析できないブロックは無視する。
        matches = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL)
        for match in matches:
            try:
                data = json.loads(match)
            except Exception:
                continue
            name = data.get("name")
            args = data.get("arguments", data.get("args", {}))
            if isinstance(args, str):
                # 引数が JSON 文字列の場合はオブジェクトへ戻す。失敗した場合は文字列のまま残す。
                try:
                    args = json.loads(args)
                except Exception:
                    pass
            if name:
                result.append({"id": "call_" + uuid.uuid4().hex, "name": name, "args": args})
        return result

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # middleware 後のメッセージと bind 済みツールを、モデルが実際に見る入力に変換する。
        hf_messages = self._convert_messages(messages)
        template_kwargs = {"tokenize": True, "add_generation_prompt": True, "return_tensors": "pt", "return_dict": True}
        if self._bound_tools:
            template_kwargs["tools"] = self._bound_tools
        inputs = self.tokenizer.apply_chat_template(hf_messages, **template_kwargs)
        device = next(self.model.parameters()).device
        moved_inputs = {}
        for key in inputs:
            value = inputs[key]
            if hasattr(value, "to"):
                moved_inputs[key] = value.to(device)
            else:
                moved_inputs[key] = value
        inputs = moved_inputs

        # 生成トークン用の余地を確保してからモデルを呼び、コンテキスト超過を防ぐ。
        inputs, original_input_length, used_input_length, truncated_tokens = truncate_generation_inputs(inputs, self.max_seq_length, self.max_new_tokens, self.truncation_head_tokens)
        input_ids = inputs["input_ids"]
        assert used_input_length + self.max_new_tokens <= self.max_seq_length, "Generation token budget exceeds MAX_SEQ_LENGTH."
        if truncated_tokens > 0:
            print(f"[truncate] input {original_input_length} -> {used_input_length}, removed={truncated_tokens}, generation_budget={self.max_new_tokens}")

        # ロールアウトでは勾配を作らずサンプリングし、生成直後の旧方策対数確率を記録する。
        with torch.inference_mode():
            outputs = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=True, temperature=self.temperature, top_p=self.top_p, pad_token_id=self.tokenizer.pad_token_id, max_length=None)
        prompt_length = int(input_ids.shape[-1])
        generated_ids = outputs[:, prompt_length:]
        old_logprobs = rollout_logprobs_for_generated(self.model, input_ids, generated_ids)
        raw_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        clean_text = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

        # 先に tool call を解析し、この model call が主エージェントの decision turn かを識別できる情報と一緒に保存する。
        # DeepAgents の summarization middleware は同じ model を内部利用するが、通常は agent tool を bind していない model call になる。
        tool_calls = self._parse_tool_calls(raw_text)
        had_bound_tools = len(self._bound_tools) > 0

        # ターン固有の入力・出力・旧方策確率を CPU 側へ移し、後の GRPO 再計算に使う。
        record_input_ids = input_ids[0].detach().cpu().tolist()
        record_output_ids = generated_ids[0].detach().cpu().tolist()
        input_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=False)
        stored_tool_calls = []

        for tool_call in tool_calls:
            stored_tool_calls.append({"id": tool_call.get("id"), "name": tool_call.get("name"), "args": tool_call.get("args", {})})

        self._records.append(LLMCallRecord(input_ids=record_input_ids, output_ids=record_output_ids, old_logprobs=old_logprobs, input_text=input_text, output_text=raw_text, clean_text=clean_text, parsed_tool_calls=stored_tool_calls, had_bound_tools=had_bound_tools, original_input_length=original_input_length, used_input_length=used_input_length, truncated_tokens=truncated_tokens))

        # ツール呼び出しなら DeepAgents に実行を委ね、それ以外は最終回答として返す。
        if tool_calls:
            ai_message = AIMessage(content="", tool_calls=tool_calls)
        else:
            ai_message = AIMessage(content=clean_text)
        del outputs
        del generated_ids
        del input_ids
        del inputs
        return ChatResult(generations=[ChatGeneration(message=ai_message)])

    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs):
        # DeepAgents の非同期呼び出しから同期生成処理へ橋渡しする。
        return self._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


# ============================================================
# 9. DeepAgents message history を decision turn に変換
# ============================================================

def canonical_tool_call(tool_name, args):
    """duplicate 判定用に tool 名と args を安定した JSON 文字列へ正規化する。"""
    data = {"name": tool_name, "args": args}
    return json.dumps(data, ensure_ascii=False, sort_keys=True, default=str)


def tool_message_is_error(message):
    """ToolMessage が error を表すかを status と本文の両方から判定する。"""
    status = getattr(message, "status", None)

    if isinstance(status, str) and status.lower() == "error":
        return True

    text = str(message.content).lower()
    error_markers = ["error:", "exception", "traceback", "failed", "failure"]
    return contains_any(text, error_markers)


def normalized_tool_call_list(tool_calls):
    """tool call ID を除外し、name + args だけで model record と AIMessage を比較できる形にする。"""
    normalized = []

    for call in tool_calls:
        normalized.append(canonical_tool_call(call.get("name"), call.get("args", {})))

    return normalized


def record_matches_ai_message(record, message):
    """
    LLMCallRecord が result["messages"] の可視 AIMessage に対応するか判定する。

    tool turn は name + args の並びを比較し、final turn は clean text を比較する。
    UUID の tool_call_id は middleware による変換の影響を避けるため照合条件に使わない。
    """
    message_calls = message.tool_calls or []

    if len(message_calls) > 0:
        record_calls = normalized_tool_call_list(record.parsed_tool_calls)
        visible_calls = normalized_tool_call_list(message_calls)
        return record_calls == visible_calls

    if len(record.parsed_tool_calls) > 0:
        return False

    message_text = ""

    if isinstance(message.content, str):
        message_text = message.content.strip()

    return record.clean_text.strip() == message_text


def collect_visible_turns(result):
    """
    result["messages"] から主エージェントの可視 assistant turn と直後の ToolMessage を作る。

    middleware が内部で行った要約生成などは result の agent conversation turn ではないため、
    ここには含まれない。
    """
    messages = result.get("messages", [])
    turns = []
    visible_ai_messages = []
    current_turn = None

    for message in messages:
        if isinstance(message, AIMessage):
            tool_calls = []
            source_calls = message.tool_calls or []

            for call in source_calls:
                tool_calls.append({"id": call.get("id"), "name": call.get("name"), "args": call.get("args", {})})

            assistant_content = ""

            if isinstance(message.content, str):
                assistant_content = message.content.strip()

            is_final = len(tool_calls) == 0
            current_turn = TurnDetail(assistant_content=assistant_content, tool_calls=tool_calls, tool_messages=[], is_final=is_final)
            turns.append(current_turn)
            visible_ai_messages.append(message)

        elif isinstance(message, ToolMessage):
            if current_turn is not None:
                current_turn.tool_messages.append(message)

    return turns, visible_ai_messages


def align_visible_records(result, all_records):
    """
    全 model call record から、主エージェントの可視 decision turn に対応する record だけを抽出する。

    DeepAgents では SummarizationMiddleware が同じ model を内部呼び出しするため、
    policy.records には agent conversation に現れない model call が混入し得る。

    第1段階:
    - tool が bind されていた record だけを main-agent candidate とする。
    - candidate 数と可視 AIMessage 数が一致し、内容も順番に一致すればそのまま採用する。

    第2段階:
    - 数が一致しない場合は全 record を順方向に走査し、各可視 AIMessage と内容が一致する record を探す。
    - 一致しない内部 record は training target から除外する。

    silent に件数だけ合わせるのではなく、tool call または final text の内容一致を必須にする。
    """
    turns, visible_ai_messages = collect_visible_turns(result)
    candidate_records = []

    for record in all_records:
        if record.had_bound_tools:
            candidate_records.append(record)

    direct_match = len(candidate_records) == len(visible_ai_messages)

    if direct_match:
        for index in range(len(visible_ai_messages)):
            if not record_matches_ai_message(candidate_records[index], visible_ai_messages[index]):
                direct_match = False
                break

    if direct_match:
        skipped_internal_calls = len(all_records) - len(candidate_records)
        return candidate_records, turns, skipped_internal_calls

    aligned_records = []
    search_start = 0

    for message_index in range(len(visible_ai_messages)):
        message = visible_ai_messages[message_index]
        matched_index = None

        for record_index in range(search_start, len(all_records)):
            record = all_records[record_index]

            if record_matches_ai_message(record, message):
                matched_index = record_index
                break

        if matched_index is None:
            raise RuntimeError(f"Visible-turn alignment failed at assistant_message={message_index}. total_model_records={len(all_records)}, visible_assistant_messages={len(visible_ai_messages)}, bound_tool_candidates={len(candidate_records)}.")

        aligned_records.append(all_records[matched_index])
        search_start = matched_index + 1

    skipped_internal_calls = len(all_records) - len(aligned_records)

    if len(aligned_records) != len(turns):
        raise RuntimeError(f"Visible-turn alignment produced inconsistent lengths: records={len(aligned_records)}, turns={len(turns)}.")

    return aligned_records, turns, skipped_internal_calls


def summarize_turns(turns):
    """trajectory 全体の最終回答、observation、tool/error/duplicate 数を集計する。"""
    final_answer = ""
    observations = []
    tool_calls = 0
    tool_errors = 0
    duplicate_calls = 0
    seen_calls = set()

    for turn in turns:
        if turn.is_final and turn.assistant_content:
            final_answer = turn.assistant_content

        for call in turn.tool_calls:
            tool_calls += 1
            key = canonical_tool_call(call.get("name"), call.get("args", {}))

            if key in seen_calls:
                duplicate_calls += 1
            else:
                seen_calls.add(key)

        for tool_message in turn.tool_messages:
            text = str(tool_message.content)
            observations.append(text)

            if tool_message_is_error(tool_message):
                tool_errors += 1

    observations_text = "\n\n".join(observations)
    return final_answer, observations_text, tool_calls, tool_errors, duplicate_calls


# ============================================================
# 10. Turn-level reward
# ============================================================

FILE_RE = re.compile(r"\b[\w.\-]+(?:/[\w.\-]+)+\.(?:py|toml|yaml|yml|json|md|sh|js|ts|rs|go)\b", re.I)


def evidence_satisfied(requirement, answer, observations):
    """回答と DeepWiki observation を合わせ、evidence requirement を簡易判定する。"""
    combined = (answer + "\n" + observations).lower()
    req = requirement.lower()

    if "readme" in req:
        return "readme" in combined

    if "2つ以上" in requirement and "ディレクトリ" in requirement:
        directory_matches = re.findall(r"\b[\w.\-]+/", combined)
        directories = set(directory_matches)
        return len(directories) >= 2

    implementation_markers = ["実ファイル", "設定ファイル", "エントリーポイント", "実装"]

    for marker in implementation_markers:
        if marker in requirement:
            return bool(FILE_RE.search(combined))

    if "利用例" in requirement:
        example_markers = ["example", "examples", "usage", "quickstart", "利用例", "使用例"]
        return contains_any(combined, example_markers)

    if "必須依存" in requirement:
        dependency_markers = ["dependencies", "requirements", "pyproject.toml", "setup.py"]

        for marker in dependency_markers:
            if marker in combined:
                return True

        return False

    if "任意" in requirement or "開発依存" in requirement:
        optional_markers = ["optional", "dev", "extras", "dependency-groups"]

        for marker in optional_markers:
            if marker in combined:
                return True

        return False

    raw_tokens = re.split(r"[、。\s]+", requirement)
    tokens = []

    for token in raw_tokens:
        if len(token) >= 2:
            tokens.append(token)

    for token in tokens:
        if token.lower() in combined:
            return True

    return False


def calculate_final_answer_reward(final_answer, observations, row):
    """
    final answer に属する reward を計算する。

    この reward は final turn の local reward にだけ配置する。
    それ以前の tool turn へは return-to-go によって将来 reward を伝播する。
    """
    spec = row["reward_spec"]
    keywords = row["must_mention"] or []
    requirements = row["evidence_requirements"] or []
    keyword_reward = 0.0
    evidence_reward = 0.0
    unsupported_reward = 0.0
    quality_reward = 0.0

    if len(keywords) > 0:
        keyword_hits = count_true_strings_in_text(final_answer.lower(), keywords)
        keyword_coverage = keyword_hits / len(keywords)
    else:
        keyword_coverage = 1.0

    keyword_reward = keyword_coverage * float(spec.get("keyword_coverage_weight", 2.0))

    if len(requirements) > 0:
        evidence_hits = 0

        for requirement in requirements:
            if evidence_satisfied(requirement, final_answer, observations):
                evidence_hits += 1

        evidence_coverage = evidence_hits / len(requirements)
    else:
        evidence_coverage = 1.0

    evidence_reward = evidence_coverage * float(spec.get("evidence_coverage_weight", 2.0))

    path_matches = FILE_RE.findall(final_answer)
    paths = set(path_matches)

    if len(paths) > 0:
        unsupported = 0

        for path in paths:
            if path not in observations:
                unsupported += 1

        unsupported_ratio = unsupported / len(paths)
        unsupported_reward = unsupported_ratio * float(spec.get("unsupported_claim_penalty", -1.0))

    if not final_answer:
        quality_reward = -1.0
    elif len(final_answer) < 100:
        quality_reward = -0.3
    else:
        quality_reward = 0.2

    conclusion_markers = ["結論", "まとめ", "要約"]

    if contains_any(final_answer, conclusion_markers):
        quality_reward += 0.1

    total_reward = keyword_reward + evidence_reward + unsupported_reward + quality_reward
    components = {"keyword": keyword_reward, "evidence": evidence_reward, "unsupported": unsupported_reward, "quality": quality_reward}
    return float(total_reward), components


def assign_turn_rewards(trajectory, row):
    """
    decision turn ごとに local reward を割り当てる。

    Tool turn:
      - 成功 tool が1件以上あれば success bonus は1回だけ。
      - tool error は件数分 penalty。
      - 既出の同一 tool+args は duplicate penalty。
      - 1 turn 内の tool call が閾値を超えた分だけ軽い penalty。

    Final turn:
      - keyword / evidence / unsupported claim / answer quality。

    success reward を tool call 数に比例させないことで、
    「無駄に tool を多く呼ぶほど得」という reward hacking を抑える。
    """
    spec = row["reward_spec"]
    seen_calls = set()
    turn_rewards = []
    aggregate_components = {"tool_success": 0.0, "tool_error": 0.0, "duplicate": 0.0, "extra_tool_calls": 0.0, "keyword": 0.0, "evidence": 0.0, "unsupported": 0.0, "quality": 0.0}

    for turn in trajectory.turns:
        components = {}
        local_reward = 0.0

        if len(turn.tool_calls) > 0:
            successful_tool_results = 0
            error_tool_results = 0
            duplicate_count = 0

            for tool_message in turn.tool_messages:
                if tool_message_is_error(tool_message):
                    error_tool_results += 1
                else:
                    successful_tool_results += 1

            for call in turn.tool_calls:
                key = canonical_tool_call(call.get("name"), call.get("args", {}))

                if key in seen_calls:
                    duplicate_count += 1
                else:
                    seen_calls.add(key)

            success_reward = 0.0

            if successful_tool_results > 0:
                success_reward = float(spec.get("tool_success_weight", 0.3))

            error_reward = error_tool_results * float(spec.get("tool_error_penalty", -1.0))
            duplicate_reward = duplicate_count * float(spec.get("duplicate_tool_call_penalty", -0.5))
            extra_call_count = max(0, len(turn.tool_calls) - MAX_FREE_TOOL_CALLS_PER_TURN)
            extra_call_reward = extra_call_count * EXTRA_TOOL_CALL_PENALTY
            local_reward = success_reward + error_reward + duplicate_reward + extra_call_reward
            components = {"tool_success": success_reward, "tool_error": error_reward, "duplicate": duplicate_reward, "extra_tool_calls": extra_call_reward}

        else:
            final_reward, final_components = calculate_final_answer_reward(trajectory.final_answer, trajectory.observations, row)
            local_reward = final_reward
            components = final_components

        turn.local_reward = float(local_reward)
        turn.reward_components = components
        turn_rewards.append(float(local_reward))

        for key in components:
            if key not in aggregate_components:
                aggregate_components[key] = 0.0

            aggregate_components[key] += float(components[key])

    trajectory.turn_rewards = turn_rewards
    trajectory.reward_components = aggregate_components
    total_reward = 0.0

    for reward in turn_rewards:
        total_reward += reward

    trajectory.reward = float(total_reward)


# ============================================================
# 11. Return-to-go と reach-aware leave-one-out advantage
# ============================================================

def compute_returns_to_go(turn_rewards, gamma):
    """
    G_t = r_t + gamma * G_{t+1} を後ろから計算する。

    final answer の品質 reward が前段の検索 action にも伝わるため、
    local reward だけを各 turn に適用するより長期 credit assignment ができる。
    """
    returns = [0.0] * len(turn_rewards)
    running_return = 0.0

    for turn_index in range(len(turn_rewards) - 1, -1, -1):
        running_return = float(turn_rewards[turn_index]) + gamma * running_return
        returns[turn_index] = running_return

    return returns


def compute_turn_level_advantages(trajectories, gamma):
    """
    真の turn-level credit assignment の中心部分。

    turn t の baseline には、その turn に実際に到達した rollout だけを使う。
    trajectory i 自身を baseline から外し、leave-one-out mean を用いる。

    A_{i,t} = G_{i,t} - mean(G_{j,t}; j != i, j reached t)

    比較相手が存在しない turn は advantage=0 とし、
    不確かな1本だけの return で policy を更新しない。
    """
    all_returns = []
    maximum_turns = 0

    for trajectory in trajectories:
        returns = compute_returns_to_go(trajectory.turn_rewards, gamma)
        trajectory.returns_to_go = returns
        all_returns.append(returns)

        if len(returns) > maximum_turns:
            maximum_turns = len(returns)

    all_advantages = []

    for returns in all_returns:
        all_advantages.append([0.0] * len(returns))

    for turn_index in range(maximum_turns):
        reached_indices = []

        for trajectory_index in range(len(all_returns)):
            if turn_index < len(all_returns[trajectory_index]):
                reached_indices.append(trajectory_index)

        if len(reached_indices) < 2:
            for trajectory_index in reached_indices:
                all_advantages[trajectory_index][turn_index] = 0.0

            continue

        for trajectory_index in reached_indices:
            baseline_sum = 0.0
            baseline_count = 0

            for other_index in reached_indices:
                if other_index == trajectory_index:
                    continue

                baseline_sum += float(all_returns[other_index][turn_index])
                baseline_count += 1

            baseline = baseline_sum / baseline_count
            advantage = float(all_returns[trajectory_index][turn_index]) - baseline

            if TURN_ADVANTAGE_CLIP is not None:
                advantage = max(-TURN_ADVANTAGE_CLIP, min(TURN_ADVANTAGE_CLIP, advantage))

            all_advantages[trajectory_index][turn_index] = advantage

    for trajectory_index in range(len(trajectories)):
        trajectories[trajectory_index].turn_advantages = all_advantages[trajectory_index]

    return all_advantages


# ============================================================
# 12. DeepAgents rollout と Turn-Level GRPO backward
# ============================================================

def run_trajectory(row):
    """1 dataset row から DeepAgents の実 multi-turn trajectory を1本生成する。"""
    FastLanguageModel.for_inference(model)
    policy = DeepAgentsPolicyModel(model=model, tokenizer=tokenizer, max_seq_length=MAX_SEQ_LENGTH, max_new_tokens=MAX_NEW_TOKENS, truncation_head_tokens=TRUNCATION_HEAD_TOKENS, temperature=ROLLOUT_TEMPERATURE, top_p=ROLLOUT_TOP_P)
    policy.clear_records()
    agent = create_deep_agent(model=policy, tools=deepwiki_tools, system_prompt="")

    # Dataset 側
    # max_tool_steps = int(row.get("max_tool_steps", 8))
    max_tool_steps = 8

    # LangGraph 側の emergency safety limit
    recursion_limit = max(128, max_tool_steps * 8 + 32)

    try:
        result = run_async_sync(agent.ainvoke({"messages": row["prompt"]}, config={"recursion_limit": recursion_limit}))
    finally:
        FastLanguageModel.for_training(model)
        model.config.use_cache = False

    all_records = list(policy.records)
    records, turns, internal_model_calls = align_visible_records(result, all_records)
    final_answer, observations, tool_calls, tool_errors, duplicate_calls = summarize_turns(turns)
    trajectory = AgentTrajectory(records=records, turns=turns, final_answer=final_answer, observations=observations, tool_calls=tool_calls, tool_errors=tool_errors, duplicate_calls=duplicate_calls, internal_model_calls=internal_model_calls)

    if internal_model_calls > 0:
        print(f"[alignment] excluded_internal_model_calls={internal_model_calls} total_records={len(all_records)} visible_turns={len(records)}")

    assign_turn_rewards(trajectory, row)
    return trajectory


def backward_turn_level_grpo(trajectory, num_trajectories, clip_epsilon=0.2, beta=0.0):
    """
    各 turn 固有の advantage を、その turn が生成した token だけに適用する。

    OOM-safe 性は Step2 と同じで、各 turn をさらに token chunk に分割し、
    chunk ごとに forward -> backward -> graph 解放を行う。
    """
    if len(trajectory.records) != len(trajectory.turn_advantages):
        raise RuntimeError(f"Turn advantage alignment failed: records={len(trajectory.records)}, advantages={len(trajectory.turn_advantages)}")

    total_tokens = 0

    for record in trajectory.records:
        total_tokens += len(record.output_ids)

    if total_tokens == 0:
        return {"loss": 0.0, "policy_loss": 0.0, "kl": 0.0, "tokens": 0, "chunks": 0, "active_turns": 0}

    total_loss_value = 0.0
    total_policy_value = 0.0
    total_kl_value = 0.0
    chunk_count = 0
    active_turns = 0

    for turn_index in range(len(trajectory.records)):
        record = trajectory.records[turn_index]
        turn_advantage = float(trajectory.turn_advantages[turn_index])
        output_length = len(record.output_ids)

        if output_length == 0:
            continue

        assert len(record.old_logprobs) == output_length, "Stored old_logprobs length does not match output_ids."

        if abs(turn_advantage) > 0.0:
            active_turns += 1

        start_index = 0

        while start_index < output_length:
            end_index = min(start_index + TRAIN_LOGPROB_CHUNK_SIZE, output_length)
            chunk_length = end_index - start_index
            current_lp = chunk_logprobs_with_grad(record, start_index, end_index)
            old_lp = torch.tensor(record.old_logprobs[start_index:end_index], dtype=current_lp.dtype, device=current_lp.device)
            advantage_tensor = torch.tensor(turn_advantage, dtype=current_lp.dtype, device=current_lp.device)

            assert current_lp.shape == old_lp.shape, "Current and old policy logprob shapes do not match."

            log_ratio = current_lp - old_lp.detach()
            ratio = torch.exp(log_ratio)
            unclipped = ratio * advantage_tensor
            clipped_ratio = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon)
            clipped = clipped_ratio * advantage_tensor
            token_policy_loss = -torch.minimum(unclipped, clipped)

            if beta > 0.0:
                ref_lp = chunk_reference_logprobs(record, start_index, end_index)
                ref_delta = ref_lp - current_lp
                token_kl = torch.exp(ref_delta) - ref_delta - 1.0
            else:
                token_kl = torch.zeros_like(token_policy_loss)

            token_loss = token_policy_loss + beta * token_kl

            # response 全 token 平均に対応させるため chunk_length / total_tokens で重み付けする。
            # 同じ turn の全 chunk には同じ turn_advantage が適用される。
            chunk_weight = chunk_length / total_tokens
            scaled_loss = token_loss.mean() * chunk_weight / num_trajectories
            scaled_loss.backward()

            total_loss_value += float(token_loss.mean().detach()) * chunk_weight
            total_policy_value += float(token_policy_loss.mean().detach()) * chunk_weight
            total_kl_value += float(token_kl.mean().detach()) * chunk_weight
            chunk_count += 1

            del current_lp
            del old_lp
            del advantage_tensor
            del log_ratio
            del ratio
            del unclipped
            del clipped_ratio
            del clipped
            del token_policy_loss
            del token_kl
            del token_loss
            del scaled_loss

            if beta > 0.0:
                del ref_lp
                del ref_delta

            start_index = end_index

    return {"loss": total_loss_value, "policy_loss": total_policy_value, "kl": total_kl_value, "tokens": total_tokens, "chunks": chunk_count, "active_turns": active_turns}


# ============================================================
# 13. オプティマイザ
# ============================================================

# 量子化済みベースモデルを除き、勾配更新が有効な LoRA パラメータのみを渡す。
trainable_parameters = []
for parameter in model.parameters():
    if parameter.requires_grad:
        trainable_parameters.append(parameter)

# 学習対象は LoRA パラメータだけなので、通常の AdamW オプティマイザで十分に小さく収まる。
optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, betas=(0.9, 0.99), weight_decay=0.0)


# ============================================================
# 14. 再現性
# ============================================================

# サンプル順序と PyTorch／CUDA の乱数系列をそろえるためにシードを設定する。
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# 15. 学習ループ
# ============================================================

global_step = 0
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    indices = list(range(len(train_dataset)))
    random.shuffle(indices)

    print("\n==============================")
    print(f"EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("==============================")

    for sample_idx in indices:
        row = train_dataset[sample_idx]
        torch.cuda.reset_peak_memory_stats()
        trajectories = []

        print(f"\n[rollout] id={row['id']} generations={NUM_GENERATIONS}")

        # 同じ policy parameter のまま G 本を収集し、全 turn advantage が確定するまで optimizer.step() しない。
        for generation_index in range(NUM_GENERATIONS):
            try:
                trajectory = run_trajectory(row)
            except torch.cuda.OutOfMemoryError as exc:
                cleanup_cuda()
                gpu_memory_snapshot("rollout OOM")
                raise RuntimeError("CUDA OOM during DeepAgents rollout. Reduce MAX_NEW_TOKENS or MAX_SEQ_LENGTH.") from exc

            trajectories.append(trajectory)
            total_truncated = 0
            max_used_input = 0

            for record in trajectory.records:
                total_truncated += record.truncated_tokens

                if record.used_input_length > max_used_input:
                    max_used_input = record.used_input_length

            print(f"  rollout={generation_index} reward={trajectory.reward:.3f} turns={len(trajectory.records)} tools={trajectory.tool_calls} errors={trajectory.tool_errors} duplicates={trajectory.duplicate_calls} internal_calls={trajectory.internal_model_calls} truncated={total_truncated} max_input={max_used_input}")
            print(f"    turn_rewards={trajectory.turn_rewards}")

        gpu_memory_snapshot("after rollouts")

        # Level-3 の核心:
        # local turn reward -> realised return-to-go -> turn 到達 rollout の leave-one-out advantage。
        compute_turn_level_advantages(trajectories, TURN_RETURN_GAMMA)

        print("  turn-level credit assignment:")

        for trajectory_index in range(len(trajectories)):
            trajectory = trajectories[trajectory_index]
            print(f"    rollout={trajectory_index} returns={trajectory.returns_to_go}")
            print(f"    rollout={trajectory_index} advantages={trajectory.turn_advantages}")

        # trajectory total reward の mean/std は診断用に残す。
        # 学習 gradient には trajectory scalar advantage を使用しない。
        rewards = torch.empty(len(trajectories), dtype=torch.float32)

        for trajectory_index in range(len(trajectories)):
            rewards[trajectory_index] = trajectories[trajectory_index].reward

        mean_reward = rewards.mean()
        std_reward = rewards.std(unbiased=False)

        cleanup_cuda()
        FastLanguageModel.for_training(model)
        model.config.use_cache = False
        model.train()
        optimizer.zero_grad(set_to_none=True)
        stats = []

        try:
            for trajectory in trajectories:
                current_stats = backward_turn_level_grpo(trajectory, len(trajectories), CLIP_EPSILON, BETA)
                stats.append(current_stats)
                cleanup_cuda()
        except torch.cuda.OutOfMemoryError as exc:
            optimizer.zero_grad(set_to_none=True)
            cleanup_cuda()
            gpu_memory_snapshot("update OOM")
            raise RuntimeError("CUDA OOM during turn-level chunked GRPO update. Reduce TRAIN_LOGPROB_CHUNK_SIZE first, then MAX_SEQ_LENGTH.") from exc

        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        avg_loss = 0.0
        avg_policy_loss = 0.0
        avg_kl = 0.0
        num_policy_tokens = 0
        num_chunks = 0
        total_active_turns = 0

        for item in stats:
            avg_loss += item["loss"]
            avg_policy_loss += item["policy_loss"]
            avg_kl += item["kl"]
            num_policy_tokens += item["tokens"]
            num_chunks += item["chunks"]
            total_active_turns += item["active_turns"]

        stats_count = max(len(stats), 1)
        avg_loss = avg_loss / stats_count
        avg_policy_loss = avg_policy_loss / stats_count
        avg_kl = avg_kl / stats_count

        total_tools = 0
        total_errors = 0
        total_duplicates = 0
        total_truncated_tokens = 0
        maximum_input_tokens = 0
        total_turns = 0
        zero_advantage_turns = 0
        total_advantage_turns = 0
        absolute_advantage_sum = 0.0
        aggregate_reward_components = {}

        for trajectory in trajectories:
            total_tools += trajectory.tool_calls
            total_errors += trajectory.tool_errors
            total_duplicates += trajectory.duplicate_calls
            total_turns += len(trajectory.turns)

            for record in trajectory.records:
                total_truncated_tokens += record.truncated_tokens

                if record.used_input_length > maximum_input_tokens:
                    maximum_input_tokens = record.used_input_length

            for advantage in trajectory.turn_advantages:
                total_advantage_turns += 1
                absolute_advantage_sum += abs(float(advantage))

                if abs(float(advantage)) < 1e-12:
                    zero_advantage_turns += 1

            for key in trajectory.reward_components:
                if key not in aggregate_reward_components:
                    aggregate_reward_components[key] = 0.0

                aggregate_reward_components[key] += float(trajectory.reward_components[key])

        avg_tools = total_tools / len(trajectories)
        avg_errors = total_errors / len(trajectories)
        avg_duplicates = total_duplicates / len(trajectories)
        avg_turns = total_turns / len(trajectories)

        if total_advantage_turns > 0:
            mean_abs_turn_advantage = absolute_advantage_sum / total_advantage_turns
            zero_advantage_ratio = zero_advantage_turns / total_advantage_turns
        else:
            mean_abs_turn_advantage = 0.0
            zero_advantage_ratio = 1.0

        print(f"[step={global_step:04d}] id={row['id']} loss={avg_loss:.5f} policy_loss={avg_policy_loss:.5f} kl={avg_kl:.5f} reward={mean_reward.item():.3f} reward_std={std_reward.item():.3f} turns={avg_turns:.2f} active_turns={total_active_turns} mean_abs_turn_adv={mean_abs_turn_advantage:.3f} zero_adv_ratio={zero_advantage_ratio:.3f} tokens={num_policy_tokens} chunks={num_chunks} tools={avg_tools:.2f} errors={avg_errors:.2f} duplicates={avg_duplicates:.2f} truncated={total_truncated_tokens} max_input={maximum_input_tokens} grad_norm={float(grad_norm):.3f}")

        if len(aggregate_reward_components) > 0:
            component_text_parts = []

            for key in aggregate_reward_components:
                component_mean = aggregate_reward_components[key] / len(trajectories)
                component_text_parts.append(f"{key}={component_mean:.3f}")

            component_text = " ".join(component_text_parts)
            print("  reward_components:", component_text)

        gpu_memory_snapshot("after optimizer step")
        cleanup_cuda()

        if global_step % SAVE_EVERY == 0:
            checkpoint_path = OUTPUT_DIR + "/checkpoint-" + str(global_step)
            model.save_pretrained(checkpoint_path)
            tokenizer.save_pretrained(checkpoint_path)
            print("saved:", checkpoint_path)


# ============================================================
# 16. 最終保存
# ============================================================

# 全エポック終了後、最終アダプタと tokenizer を指定した出力先へ保存する。
model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print("Training finished:", FINAL_PATH)